# Inspect alphaZero

## First

### imports and args

In [ ]:
import sys
sys.path.append('..')
sys.path.append('../..')
from Arena import Arena
from KalahGame import KalahGame
from KalahLogic import Board, connect
from utils import *
from MCTS import MCTS  
from KalahPlayers import minmax_tobi, minmax_vince
import time
import multiprocessing
from Coach import Coach
from pytorch.NNetWrapper import NNetWrapper
from pytorch.KalahNNet import KalahNNet
import numpy as np


In [ ]:
args = dotdict({
    'numIters': 1000,
    'numEps': 1,              # Number of complete self-play games to simulate during a new iteration.
    'tempThreshold': 15,        #
    'updateThreshold': 0.6,     # During arena playoff, new neural net will be accepted if threshold or more of games are won.
    'maxlenOfQueue': 200000,    # Number of game examples to train the neural networks.
    'numMCTSSims': 4000,          # Number of games moves for MCTS to simulate.
    'arenaCompare': 40,         # Number of games to play during arena play to determine if new net will be accepted.
    'cpuct': 1,

    'checkpoint': '../best_models',
    'load_model': False,
    'load_folder_file': ('../../best_models/','best.pth.tar'),
    'numItersForTrainExamplesHistory': 20,

})

### inspect

In [ ]:
def inspect_train_samples():
    from Coach import Coach 
    c = Coach(KalahGame(), NNetWrapper(KalahGame()), args)
    c.loadTrainExamples()
    return c.trainExamplesHistory

In [ ]:
history = inspect_train_samples()

In [ ]:
print(len(history)) # different iterations of NN 
print(len(history[0])) # dofferent gamesstates for play with one NN
print(len(history[0][0])) # Board, pi, v
game = 104
print(history[1][game][0]) # Board
print(history[1][game][1]) # pi    
print(history[1][game][2]) # v

### try optimisation of MCTS

In [17]:
mcts = MCTS(KalahGame(), NNetWrapper(KalahGame()), args)
g = KalahGame()
nnet = NNetWrapper(g)
nnet.load_checkpoint(folder=args.load_folder_file[0], filename='best_64.pth.tar')
player = MCTS(g, nnet, dotdict({ 'numMCTSSims': 4000, 'cpuct': 1.0 }))

board = KalahGame().initial_board
start = time.time()
player.getActionProb(board, 1)
print(time.time() - start)

1.3691470623016357


## Second

### build New NN

#### imports

In [19]:
import sys
sys.path.append('..')
from utils import *

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim


#### conv Net from original

In [20]:
class KalahNNet_conv(nn.Module):
    def __init__(self, game, args):
        # game params
        self.board_x, self.board_y = game.getBoardSize()
        self.action_size = game.getActionSize()
        self.args = args

        super(KalahNNet_conv, self).__init__()
        self.conv1 = nn.Conv2d(1, args.num_channels, 2, stride=1, padding=1)
        self.conv2 = nn.Conv2d(args.num_channels, args.num_channels, 2, stride=1, padding=1)
        self.conv3 = nn.Conv2d(args.num_channels, args.num_channels, 2, stride=1)
        self.conv4 = nn.Conv2d(args.num_channels, args.num_channels, 2, stride=1)

        self.bn1 = nn.BatchNorm2d(args.num_channels)
        self.bn2 = nn.BatchNorm2d(args.num_channels)
        self.bn3 = nn.BatchNorm2d(args.num_channels)
        self.bn4 = nn.BatchNorm2d(args.num_channels)

        self.fc1 = nn.Linear(args.num_channels*(self.board_x)*(self.board_y), 1024)
        self.fc_bn1 = nn.BatchNorm1d(1024)

        self.fc2 = nn.Linear(1024, 512)
        self.fc_bn2 = nn.BatchNorm1d(512)

        self.fc3 = nn.Linear(512, self.action_size)

        self.fc4 = nn.Linear(512, 1)

    def forward(self, s):
        #                                                           s: batch_size x board_x x board_y
        s = s.view(-1, 1, self.board_x, self.board_y)                # batch_size x 1 x board_x x board_y
        s = F.relu(self.bn1(self.conv1(s)))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.bn2(self.conv2(s)))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.bn3(self.conv3(s)))                          # batch_size x num_channels x (board_x-2) x (board_y-2)
        s = F.relu(self.bn4(self.conv4(s)))                          # batch_size x num_channels x (board_x-4) x (board_y-4)
        s = s.view(-1, self.args.num_channels*(self.board_x)*(self.board_y))

        s = F.dropout(F.relu(self.fc_bn1(self.fc1(s))), p=self.args.dropout, training=self.training)  # batch_size x 1024
        s = F.dropout(F.relu(self.fc_bn2(self.fc2(s))), p=self.args.dropout, training=self.training)  # batch_size x 512

        pi = self.fc3(s)                                                                         # batch_size x action_size
        v = self.fc4(s)                                                                          # batch_size x 1

        return F.log_softmax(pi, dim=1), torch.tanh(v)

#### batch normalisation bigger net 

In [25]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features):
        super(ResidualBlock, self).__init__()
        self.fc = nn.Linear(in_features, in_features)
        self.bn = nn.BatchNorm1d(in_features)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn(self.fc(x)))
        out = self.bn(self.fc(out))
        out += residual
        return F.relu(out)

class KalahNNet_128res(nn.Module):
    def __init__(self, game, args):
        # game params
        self.board_x, self.board_y = game.getBoardSize()
        self.action_size = game.getActionSize()
        self.args = args

        super(KalahNNet_128res, self).__init__()
        self.fc1 = nn.Linear(self.board_x * self.board_y, 128)
        self.bn1 = nn.BatchNorm1d(128)

        # 3 Residual Blocks
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(128)

        self.fc2 = nn.Linear(128, self.action_size)
        self.fc3 = nn.Linear(128, 1)

    def forward(self, s):
        #                                                           s: batch_size x board_x x board_y
        s = s.view(-1, self.board_x * self.board_y)                # batch_size x (board_x * board_y)
        s = F.relu(self.bn1(self.fc1(s)))                          # batch_size x 128

        s = self.res1(s)                                           # batch_size x 128
        s = self.res2(s)                                           # batch_size x 128
        s = self.res3(s)                                           # batch_size x 128

        pi = self.fc2(s)                                           # batch_size x action_size
        v = self.fc3(s)                                            # batch_size x 1

        return F.log_softmax(pi, dim=1), torch.tanh(v)


### train on previous examples

In [26]:
import logging
import coloredlogs
from random import shuffle


nnet_args = dotdict({
    'lr': 0.001,
    'dropout': 0.3,
    'epochs': 10,
    'batch_size': 64,
    'cuda': torch.cuda.is_available(),
    'num_channels': 4,
})

log = logging.getLogger(__name__)

def main():
    g = KalahGame()
    nnet = NNetWrapper(g)
    nnet.nnet = KalahNNet_128res(g, nnet_args)
    history = inspect_train_samples()
    trainExamples = []
    for e in history:
        trainExamples.extend(e)
    shuffle(trainExamples)
    nnet.train(trainExamples)
    nnet.save_checkpoint(folder="../../best_models/", filename='kalah_128res.pth.tar')


main()

EPOCH ::: 1


Training Net: 100%|██████████| 3634/3634 [00:09<00:00, 369.50it/s, Loss_pi=1.33e+00, Loss_v=6.06e-01]


EPOCH ::: 2


Training Net: 100%|██████████| 3634/3634 [00:09<00:00, 377.28it/s, Loss_pi=1.11e+00, Loss_v=5.94e-01]


EPOCH ::: 3


...

EPOCH ::: 9


Training Net: 100%|██████████| 3634/3634 [00:09<00:00, 366.65it/s, Loss_pi=9.39e-01, Loss_v=5.75e-01]


EPOCH ::: 10


Training Net: 100%|██████████| 3634/3634 [00:10<00:00, 356.35it/s, Loss_pi=9.34e-01, Loss_v=5.75e-01]


Checkpoint Directory exists! 


In [27]:

g = KalahGame()
nnet = NNetWrapper(g)
nnet.nnet = KalahNNet_conv(g, nnet_args)
nnet.load_checkpoint(folder=args.load_folder_file[0], filename='kalah_conv.pth.tar')
player1 = MCTS(g, nnet, args)

pnet = NNetWrapper(g)
pnet.nnet = KalahNNet_128res(g, nnet_args)
pnet.load_checkpoint(folder=args.load_folder_file[0], filename='kalah_128res.pth.tar')
player2 = MCTS(g, pnet, args)

arena = Arena(player1=lambda x: np.argmax(player1.getActionProb(x, temp=0)), 
            player2=lambda x: np.argmax(player2.getActionProb(x, temp=0)),
            game=g, 
            display=g.display)

minmax_wins, alpha_zero_wins, draws = arena.playGames(2, verbose=True)
print("minmax_wins", minmax_wins)
print("alpha_zero_wins", alpha_zero_wins)
print("draws", draws)

Arena.playGames (1):   0%|          | 0/1 [00:00<?, ?it/s]

Turn  1 Player  1
0 | 8 8 8 8 8 8 8 8 |
  | 8 8 8 8 8 8 8 8 | 0

Turn  2 Player  -1
1 | 9 9 9 9 9 9 0 8 |
  | 9 8 8 8 8 8 8 8 | 0

Turn  3 Player  1
1 | 9  10 10 10 10 10 1  9  |
  | 9  8  8  8  8  8  8  0  | 1 

Turn  4 Player  -1
2 | 10 11 11 11 11 11 2  0  |
  | 10 8  8  8  8  8  8  0  | 1 

Turn  5 Player  1
2 | 10 11 11 11 11 11 3  1  |
  | 10 8  0  9  9  9  9  1  | 2 

Turn  6 Player  -1
3 | 11 12 0  11 11 11 3  1  |
  | 11 9  1  10 10 10 10 2  | 2 

Turn  7 Player  1
3 | 11 12 0  11 11 11 3  2  |
  | 11 9  1  10 10 10 10 0  | 3 

Turn  8 Player  -1
3 | 11 12 0  12 12 12 0  2  |
  | 11 9  1  10 10 10 10 0  | 3 

Turn  9 Player  1
3 | 11 12 0  12 12 13 1  3  |
  | 0  10 2  11 11 11 11 1  | 4 

Turn  10 Player  -1
4 | 12 0  0  12 12 13 2  4  |
  | 1  11 3  12 12 12 12 2  | 4 

Turn  11 Player  1
4 | 12 0  0  12 12 13 2  5  |
  | 1  11 3  12 12 12 12 0  | 5 

Turn  12 Player  -1
8 | 12 0  0  13 13 14 3  0  |
  | 1  11 0  12 12 12 12 0  | 5 

Turn  13 Player  1
8 | 12 0  0  13 13 14 

Arena.playGames (1): 100%|██████████| 1/1 [03:06<00:00, 186.96s/it]


Game over: Turn  93 Result  0.0001
64 | 0 0 0 0 0 0 0 0 |
   | 0 0 0 0 0 0 0 0 | 64



Arena.playGames (2):   0%|          | 0/1 [00:00<?, ?it/s]

Turn  1 Player  1
0 | 8 8 8 8 8 8 8 8 |
  | 8 8 8 8 8 8 8 8 | 0

Turn  2 Player  -1
1 | 9 9 9 0 8 8 8 8 |
  | 9 9 9 9 8 8 8 8 | 0

Turn  3 Player  1
1 | 9  9  10 1  9  9  9  9  |
  | 9  9  9  9  8  8  0  9  | 1 

Turn  4 Player  -1
2 | 10 10 11 2  10 10 10 0  |
  | 10 9  9  9  8  8  0  9  | 1 

Turn  5 Player  1
2 | 10 10 11 2  10 11 11 1  |
  | 10 9  0  10 9  9  1  10 | 2 

Turn  6 Player  -1
3 | 11 11 12 3  11 12 0  1  |
  | 11 10 1  11 9  9  1  10 | 2 

Turn  7 Player  1
3 | 11 11 12 3  11 13 1  2  |
  | 0  11 2  12 10 10 2  11 | 3 

Turn  8 Player  -1
4 | 12 12 0  3  11 13 1  3  |
  | 1  12 3  13 11 11 3  12 | 3 

Turn  9 Player  1
4 | 12 12 0  3  11 13 1  4  |
  | 1  12 3  13 11 11 0  13 | 4 

Turn  10 Player  -1
4 | 12 12 0  4  12 14 2  0  |
  | 1  12 3  13 11 11 0  13 | 4 

Turn  11 Player  1
4 | 13 13 1  5  13 15 3  1  |
  | 1  12 3  13 11 0  1  14 | 5 

Turn  12 Player  -1
4 | 13 13 1  5  13 15 4  0  |
  | 1  12 3  13 11 0  1  14 | 5 

Turn  13 Player  1
4 | 13 13 1  5  13 0  

Arena.playGames (2): 100%|██████████| 1/1 [04:30<00:00, 270.80s/it]

Game over: Turn  131 Result  1
64 | 0 0 0 0 0 0 0 0 |
   | 0 0 0 0 0 1 0 0 | 63

minmax_wins 0
alpha_zero_wins 1
draws 1
